# AXIOM — Colab runbook

**Before running anything:**
1. Runtime → Change runtime type → **A100 GPU** (Pro+).
2. Zip the project locally and upload `axiom.zip` to your Google Drive (MyDrive root):
   ```bash
   cd ~/Desktop && zip -r axiom.zip axiom -x '*/.venv*' '*/__pycache__/*' '*/.git/*'
   ```

Code runs from fast local `/content`; **all outputs go to Drive** (`AXIOM_DATA`/`AXIOM_EXP`) so completed stages survive disconnects. After a disconnect, re-run cells 1–5, then use the **resume-status** cell to see where to continue.

In [ ]:
# 1. Confirm the GPU (want an A100)
!nvidia-smi

In [ ]:
# 2. Mount Drive (for the zip + persistent outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Unzip the project into fast local storage
!rm -rf /content/axiom && unzip -q /content/drive/MyDrive/axiom.zip -d /content
%cd /content/axiom

In [ ]:
# 4. Install. vLLM FIRST (it pins a compatible torch), then the project.
!pip install -q vllm
!pip install -q -e . -r requirements.txt

In [ ]:
# 5. Verify the install landed (catches the usual Colab torch/vLLM mismatch up front).
# If anything is MISSING or cuda is False: Runtime -> Restart session, then re-run from cell 3.
import torch, importlib.metadata as M
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', gpu)
for pkg in ['vllm', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate']:
    try:
        print(pkg.ljust(13), M.version(pkg))
    except Exception as e:
        print(pkg.ljust(13), 'MISSING', e)
assert torch.cuda.is_available(), 'No CUDA visible. Set Runtime -> A100 and re-run from cell 3.'

In [ ]:
# 6. Secrets, persistent output paths, and the shared run-limit (Colab doesn't auto-load .env).
import os
os.environ['HF_TOKEN'] = ''            # REQUIRED — your HuggingFace token
os.environ['AXIOM_DATA'] = '/content/drive/MyDrive/axiom_out/data'
os.environ['AXIOM_EXP']  = '/content/drive/MyDrive/axiom_out/experiments'
# os.environ['JUDGE_API_KEY'] = ''     # optional — enables the paid commonsense judge (else free NLI proxy)

# Small first pass so each stage finishes in one session. Delete the limits for the full run.
L = 'model=qwen2_5_0_5b distill.source.limit=300 eval.limit=200 wandb.enabled=false'
assert os.environ['HF_TOKEN'], 'Set HF_TOKEN above before continuing.'

In [ ]:
# 7. Resume status — after a reconnect, continue from the first '-- missing' stage.
import glob
D, E = os.environ['AXIOM_DATA'], os.environ['AXIOM_EXP']
def show(label, pattern):
    hits = glob.glob(pattern)
    print(label.ljust(13), 'OK  ' if hits else '--  missing', hits[:1])
for label, pat in [
    ('traces',     D + '/traces/*.jsonl'),
    ('compressed', D + '/compressed/*.jsonl'),
    ('sft ckpt',   E + '/sft/*'),
    ('foundry',    D + '/prm/*/labels.jsonl'),
    ('xdprm ckpt', E + '/xdprm/xdprm.pt'),
    ('grpo ckpt',  E + '/grpo/*'),
    ('eval json',  E + '/eval/results.json'),
]:
    show(label, pat)

## G0 — sanity (no training, ~minutes)

In [ ]:
!python -m pytest -m 'not slow' -q
!python scripts/07_eval.py eval=smoke wandb.enabled=false

## Track 1 — first real numbers (base + SFT, no PRM)
For the headline model add `model=phi4_mini` to **every** command (keep it consistent across all stages).

In [ ]:
# Prep: download data, build SFT traces, sparse-compress (fast, CPU-light)
!python scripts/00_download_data.py
!python scripts/01_build_traces.py {L}
!python scripts/02_compress.py {L}

In [ ]:
# QLoRA SFT (the slow Track-1 stage)
!python scripts/03_sft.py {L}

In [ ]:
# Evaluate -> base + sft rows are real numbers now
!python scripts/07_eval.py {L}
import json
print(json.dumps(json.load(open(os.environ['AXIOM_EXP'] + '/eval/results.json')), indent=2))

## Track 2 — PRM + GRPO (the headline)
`05_prm_train` runs the **G2 gate** and RAISES `GateError` if AUC < 0.7 / heads too correlated / ECE too high. If it raises, **stop** — the fix is in the foundry, not GRPO. Share the gate line (`auc=… max_head_corr=… ece=…`).

In [ ]:
# Label foundry (deterministic; reuses one vLLM pool)
!python scripts/04_prm_label.py {L}

In [ ]:
# Train XD-PRM + run the G2 gate (this is the make-or-break cell)
!python scripts/05_prm_train.py wandb.enabled=false

In [ ]:
# GRPO, then the full matrix
!python scripts/06_grpo.py {L}
!python scripts/07_eval.py {L}
print(json.dumps(json.load(open(os.environ['AXIOM_EXP'] + '/eval/results.json')), indent=2))

### Notes
- **Disconnected?** Re-run cells 1–7, check the resume-status cell, continue from the first missing stage — finished artifacts are on Drive.
- **Other domains:** add `data=commonsenseqa` (or `openbookqa`/`arc_challenge`) to Track-1/2 commands to build + label those; the judge only fires on commonsense/science.
- **Full run:** delete the limits in `L` (cell 6) once a small pass is clean end-to-end.